# Approval Gate with a Skill — Email Drafter

This notebook shows how to combine the framework's built-in approval gate
with a skill run using `run_with_skill(with_approval=True)`.

The `email-drafter` skill drafts a reply to an email. Before that draft is
accepted as the task result, the framework pauses and asks the operator to
approve or reject it — the same `request_approval` mechanic from `ch08.ipynb`
Example 2, now applied at the skill level.

## What a reader learns

- `run_with_skill()` accepts `with_approval=True`, wiring the end-of-loop
  approval gate into any skill-activated run
- The skill itself stays simple — it focuses on the task; the gate is
  the caller's concern
- Contrasts with `skill_with_human_input.ipynb`, where `HumanInputTool`
  is embedded inside the skill to collect preferences mid-execution

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama
installed on your local machine and have its LLM hosting service running.
To download Ollama, follow the instructions found on this page:
https://ollama.com/download. After downloading and installing Ollama, you
can start a service by opening a terminal and running `ollama serve`.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

In [ ]:
model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

In [ ]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)
agent = LLMAgent(llm=llm)

In [ ]:
EMAIL = """\
From: sarah.chen@acmecorp.com
To: alex@mycompany.com
Subject: Re: Q3 Partnership Proposal

Hi Alex,

Thanks for sending over the Q3 partnership proposal last week. We reviewed it
with the team and we're excited about the direction.

A few things we'd like to clarify before signing:

1. Revenue share — the proposal mentions 20% for the first year, but our
   standard agreement starts at 15%. Is there flexibility here?
2. The exclusivity clause covers our North America region, but we'd need it
   limited to the retail vertical only.
3. Start date — we're targeting September 1. Does that work on your end?

If these points can be addressed, we'd like to move quickly. Can we get on a
call this week to align?

Best,
Sarah
Sarah Chen | Head of Partnerships | Acme Corp
"""

## Running the Skill

Passing `with_approval=True` to `run_with_skill()` adds the end-of-loop
approval gate to the skill run. Once the agent has drafted the reply, the
framework will pause and ask you to **approve** or **reject** before the
result is finalised.

In [ ]:
result = await agent.run_with_skill(
    "email-drafter",
    prompt=f"Draft a reply to the following email:\n\n{EMAIL}",
    with_approval=True,
)
print(result.content)